In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import random
import concurrent.futures

print("--")
# Root directory containing all videos
merged_folder = 'Merged'

# Directory to save extracted landmarks
landmark_dir = "face_landmarks"
os.makedirs(landmark_dir, exist_ok=True)

# Directory to save images with landmarks
landmark_images_dir = "landmark_images"
os.makedirs(landmark_images_dir, exist_ok=True)

def process_video(video_path, video_name):
    print(f"Processing {video_path}")
    landmarks_data = []

    # Check if the video has already been processed by looking for the CSV file
    output_csv_path = os.path.join(landmark_dir, f"{video_name}.csv")
    if os.path.exists(output_csv_path):
        print(f"Video {video_name} already processed. Skipping...")
        return

    # Initialize MediaPipe Face Mesh inside each process
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    # Open the video
    cap = cv2.VideoCapture(video_path)

    frame_index = 0
    random_frame_index = random.randint(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1)
    saved_image = False

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process face landmarks
        face_results = face_mesh.process(frame_rgb)

        if face_results.multi_face_landmarks:
            for face_landmarks in face_results.multi_face_landmarks:
                # Extract x, y coordinates for all 468 landmarks
                for i, landmark in enumerate(face_landmarks.landmark):
                    landmarks_data.append({
                        "video": video_name,
                        "frame": frame_index,
                        "landmark_index": i,
                        "x": landmark.x,
                        "y": landmark.y
                    })

                # Save an image with landmarks on the random frame
                if frame_index == random_frame_index and not saved_image:
                    for landmark in face_landmarks.landmark:
                        x = int(landmark.x * frame.shape[1])
                        y = int(landmark.y * frame.shape[0])
                        cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)
                    output_image_path = os.path.join(landmark_images_dir, f"{video_name}.jpg")
                    cv2.imwrite(output_image_path, frame)
                    print(f"Landmark image saved for {video_name} at {output_image_path}")
                    saved_image = True

        frame_index += 1

    # Save the landmarks to a CSV file
    landmarks_df = pd.DataFrame(landmarks_data)
    landmarks_df.to_csv(output_csv_path, index=False)
    print(f"Landmarks saved for {video_name} in {output_csv_path}")

    cap.release()
    face_mesh.close()  # Close the face_mesh model after processing

def process_all_videos():
    # Collect all video files in the merged folder
    print("Collecting")
    video_files = []
    for root, _, files in os.walk(merged_folder):
        for file_name in files:
            if file_name.endswith(('.mp4', '.mov', '.MOV')):
                video_path = os.path.join(root, file_name)
                video_name, _ = os.path.splitext(file_name)
                video_files.append((video_path, video_name))
                
    video_files = video_files[20:40]
    # Use ThreadPoolExecutor instead of ProcessPoolExecutor for Jupyter compatibility
    with concurrent.futures.ThreadPoolExecutor() as executor:
        print("Executing")
        futures = [executor.submit(process_video, p[0], p[1]) for p in video_files]
        for future in concurrent.futures.as_completed(futures):
            print(future.result())

# Protect entry point and execute process_all_videos if this script is run directly
if __name__ == '__main__':
    print("Calling")
    process_all_videos()
